# Q4 Pulse 3: Binary categories from the split conversations

**Input:** `05_Q4_3_conversations_wide_deid.csv`, one row per participant, with each turn in its own column (`human_1`, `ai_1`, `human_2`, ...).

Every variable is built by reading the turn text itself each time the notebook runs, so the groups always reflect the current text in the file.

**Output:** `data/processed/06_Q4_3_binary_categories.xlsx`

| Sheet | Contents |
|---|---|
| `binary_vars` | `pid` plus the four binary variables, one row per participant |
| `audit` | What each variable was built from (matched disease terms, matched medication names, counts, lengths), for checking the coding |
| `codebook` | Definition, coding rule and counts for each variable, plus the median used for the length split |

Section 7 adds a counts table and a pie chart for each variable.

**Variables**

| Variable | 1 | 0 | Missing |
|---|---|---|---|
| `human_mentions_disease` | Participant names a disease or condition in any of their turns | No condition named | No participant text |
| `human_multi_turn` | Participant wrote more than one turn | Participant wrote one turn | No participant turns |
| `ai_multi_med` | AI names more than one distinct medication | AI names exactly one medication | AI names no medication (not applicable) |
| `human_1_long` | First participant message longer than the median | At or below the median | No first message |

The disease and medication variables are built from word lists in **Section 2**. Word lists are transparent and easy to extend, but they only catch what is on the list, so check the `audit` sheet and add anything the lists miss.

## 0. Packages

In [ ]:
import importlib
import importlib.util
import subprocess
import sys

# import name : install name
REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "matplotlib": "matplotlib",
}

missing = [pip_name for mod, pip_name in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    importlib.invalidate_caches()

for mod in REQUIRED_PACKAGES:
    module = importlib.import_module(mod)
    print(f"  {mod:<10} {getattr(module, '__version__', '')}")

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font

pd.set_option("display.max_colwidth", 120)

## 1. Paths and data

In [ ]:
PROJECT_ROOT = Path(r"C:\path\to\Q4_3_Pulse")

# ---- Check this filename and folder against your 05 output ----
IN_FILE = PROJECT_ROOT / "data" / "processed" / "05_Q4_3_conversations_wide_deid.csv"
OUT_FILE = PROJECT_ROOT / "data" / "processed" / "06_Q4_3_binary_categories.xlsx"

assert IN_FILE.exists(), f"Input not found: {IN_FILE}"
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
ID_COL = "pid"

wide = pd.read_csv(IN_FILE, dtype=str, encoding="utf-8-sig", keep_default_na=False, na_values=[""])

# Turn text columns: human_1, human_2, ... and ai_0, ai_1, ... (not the _n_words / _notes columns)
HUMAN_COLS = sorted([c for c in wide.columns if re.fullmatch(r"human_\d+", c)], key=lambda c: int(c.split("_")[1]))
AI_COLS = sorted([c for c in wide.columns if re.fullmatch(r"ai_\d+", c)], key=lambda c: int(c.split("_")[1]))

for c in ["n_turns", "n_human_turns", "human_1_n_words", "human_1_n_chars"]:
    wide[c] = pd.to_numeric(wide[c], errors="coerce")

print(f"{len(wide)} participants")
print(f"human turn columns: {HUMAN_COLS[0]} to {HUMAN_COLS[-1]} ({len(HUMAN_COLS)})")
print(f"ai turn columns:    {AI_COLS[0]} to {AI_COLS[-1]} ({len(AI_COLS)})")

## 2. Word lists and settings

Write every term in lowercase. Matching ignores case, matches whole words only, and treats an apostrophe as optional, so `crohn's` also matches `crohns`.

**Diseases and conditions.** The list covers named diseases and diagnoses, not symptoms. Words like *headache*, *rash*, *fever* or *pain* are left out on purpose, so "I have a headache" is coded 0. Move a symptom into the list if your definition should include it.

- `DISEASE_TERMS`: matched in any case.
- `DISEASE_ACRONYMS`: matched only in capitals, because some are also everyday words (`MS`, `ADD`, `AIDS`).
- `DISEASE_SUFFIXES`: also catches medical words by their ending (*-itis*, *-osis*, ...), so misspellings and conditions missing from the list are still picked up. `SUFFIX_EXCLUDE` blocks everyday words with the same endings.

In [ ]:
DISEASE_TERMS = [
    # metabolic, heart, blood
    "diabetes", "prediabetes", "pre-diabetes", "type 1 diabetes", "type 2 diabetes", "gestational diabetes",
    "hypertension", "high blood pressure", "low blood pressure", "hypotension", "high cholesterol",
    "heart disease", "heart attack", "heart failure", "afib", "atrial fibrillation", "arrhythmia",
    "stroke", "blood clot", "dvt", "anemia", "sickle cell", "hemophilia", "obesity",
    "hypothyroidism", "hyperthyroidism", "hashimoto's", "graves disease", "thyroid disease",
    # lungs, infections
    "asthma", "copd", "emphysema", "pneumonia", "bronchitis", "tuberculosis", "covid", "covid-19",
    "coronavirus", "long covid", "flu", "influenza", "common cold", "rsv", "strep", "strep throat",
    "sinus infection", "ear infection", "pink eye", "uti", "urinary tract infection", "kidney infection",
    "yeast infection", "bacterial vaginosis", "shingles", "chickenpox", "measles", "mumps", "mono",
    "mononucleosis", "lyme disease", "hiv", "herpes", "hpv", "chlamydia", "gonorrhea", "syphilis",
    "std", "sti", "sepsis", "scabies", "lice", "ringworm", "norovirus", "food poisoning", "stomach bug",
    # digestive, kidney, liver
    "ibs", "irritable bowel syndrome", "crohn's", "crohn's disease", "celiac", "celiac disease",
    "gerd", "acid reflux", "reflux", "ulcer", "stomach ulcer", "hemorrhoids", "gallstones",
    "kidney stones", "kidney disease", "ckd", "fatty liver", "liver disease", "lactose intolerance",
    # cancer
    "cancer", "breast cancer", "prostate cancer", "skin cancer", "lung cancer", "colon cancer",
    "tumor", "leukemia", "lymphoma", "melanoma",
    # joints, nerves, pain conditions
    "arthritis", "osteoarthritis", "rheumatoid arthritis", "gout", "lupus", "fibromyalgia",
    "osteoporosis", "scoliosis", "sciatica", "carpal tunnel", "plantar fasciitis", "herniated disc",
    "hernia", "migraine", "migraines", "epilepsy", "seizure disorder", "multiple sclerosis",
    "parkinson's", "alzheimer's", "dementia", "neuropathy", "concussion", "vertigo", "tinnitus",
    "chronic fatigue syndrome", "ehlers-danlos", "pots",
    # skin, eyes
    "eczema", "psoriasis", "rosacea", "acne", "cellulitis", "hives", "glaucoma", "cataracts",
    "macular degeneration",
    # reproductive
    "pcos", "polycystic ovary syndrome", "endometriosis", "fibroids", "preeclampsia",
    "erectile dysfunction",
    # mental health, sleep, development
    "depression", "anxiety", "anxiety disorder", "panic disorder", "ptsd", "ocd", "adhd", "autism",
    "bipolar", "bipolar disorder", "schizophrenia", "eating disorder", "insomnia", "sleep apnea",
    "narcolepsy",
    # allergy, immune
    "allergy", "allergies", "food allergy", "celiac", "anaphylaxis", "autoimmune disease",
    "cystic fibrosis",
]

# Matched only when written in capitals
DISEASE_ACRONYMS = ["MS", "ADD", "AIDS", "RA", "BV", "CF", "EDS", "CHF", "CAD", "TBI"]

# Medical word endings, plus generic condition words that follow a name ("Graves' disease")
DISEASE_SUFFIXES = ["itis", "osis", "emia", "oma", "pathy", "algia"]
DISEASE_HEAD_WORDS = ["disease", "syndrome", "disorder", "deficiency", "infection"]

SUFFIX_EXCLUDE = {
    "diagnosis", "prognosis", "osmosis", "symbiosis", "hypnosis", "aroma", "coma", "diploma",
    "comma", "soma", "stoma", "trauma", "empathy", "sympathy", "apathy", "telepathy", "nostalgia",
    "academia", "neurosis",
}

**Medications.** `MED_TERMS` maps each name to one standard name, so a brand and its generic count once (*Advil* and *ibuprofen* are both `ibuprofen`). Add entries as `"name": "standard name"`.

- `MED_SUFFIXES` catches generic drug names by their ending (*-pril*, *-statin*, *-cillin*, ...) when they are not on the list.
- `INCLUDE_SUPPLEMENTS`: vitamins and supplements such as melatonin count as medications only when set to `True`.
- `INCLUDE_DRUG_CLASSES`: general class words (*antibiotics*, *antihistamines*, *NSAIDs*) count as medications only when set to `True`. They are off by default because "an NSAID like ibuprofen" would otherwise count twice.

In [ ]:
MED_TERMS = {
    # pain, fever, allergy, cold
    "ibuprofen": "ibuprofen", "advil": "ibuprofen", "motrin": "ibuprofen",
    "acetaminophen": "acetaminophen", "tylenol": "acetaminophen", "paracetamol": "acetaminophen",
    "aspirin": "aspirin", "naproxen": "naproxen", "aleve": "naproxen",
    "diphenhydramine": "diphenhydramine", "benadryl": "diphenhydramine",
    "cetirizine": "cetirizine", "zyrtec": "cetirizine", "loratadine": "loratadine", "claritin": "loratadine",
    "fexofenadine": "fexofenadine", "allegra": "fexofenadine", "hydroxyzine": "hydroxyzine",
    "pseudoephedrine": "pseudoephedrine", "sudafed": "pseudoephedrine", "phenylephrine": "phenylephrine",
    "dextromethorphan": "dextromethorphan", "robitussin": "dextromethorphan",
    "guaifenesin": "guaifenesin", "mucinex": "guaifenesin", "dayquil": "dayquil", "nyquil": "nyquil",
    "fluticasone": "fluticasone", "flonase": "fluticasone", "montelukast": "montelukast", "singulair": "montelukast",
    # stomach
    "omeprazole": "omeprazole", "prilosec": "omeprazole", "esomeprazole": "esomeprazole", "nexium": "esomeprazole",
    "pantoprazole": "pantoprazole", "famotidine": "famotidine", "pepcid": "famotidine", "tums": "calcium carbonate",
    "loperamide": "loperamide", "imodium": "loperamide", "pepto-bismol": "bismuth subsalicylate",
    "pepto": "bismuth subsalicylate", "miralax": "polyethylene glycol", "docusate": "docusate",
    "ondansetron": "ondansetron", "zofran": "ondansetron",
    # diabetes, weight, heart
    "metformin": "metformin", "insulin": "insulin", "ozempic": "semaglutide", "wegovy": "semaglutide",
    "rybelsus": "semaglutide", "semaglutide": "semaglutide", "mounjaro": "tirzepatide", "zepbound": "tirzepatide",
    "tirzepatide": "tirzepatide", "trulicity": "dulaglutide", "jardiance": "empagliflozin", "farxiga": "dapagliflozin",
    "lisinopril": "lisinopril", "losartan": "losartan", "amlodipine": "amlodipine", "metoprolol": "metoprolol",
    "atenolol": "atenolol", "propranolol": "propranolol", "hydrochlorothiazide": "hydrochlorothiazide",
    "furosemide": "furosemide", "lasix": "furosemide", "spironolactone": "spironolactone",
    "atorvastatin": "atorvastatin", "lipitor": "atorvastatin", "rosuvastatin": "rosuvastatin", "crestor": "rosuvastatin",
    "simvastatin": "simvastatin", "warfarin": "warfarin", "eliquis": "apixaban", "apixaban": "apixaban",
    "xarelto": "rivaroxaban", "clopidogrel": "clopidogrel", "plavix": "clopidogrel",
    # thyroid, steroids, breathing
    "levothyroxine": "levothyroxine", "synthroid": "levothyroxine",
    "prednisone": "prednisone", "prednisolone": "prednisolone", "methylprednisolone": "methylprednisolone",
    "medrol": "methylprednisolone", "hydrocortisone": "hydrocortisone", "albuterol": "albuterol",
    "ventolin": "albuterol", "epinephrine": "epinephrine", "epipen": "epinephrine",
    # antibiotics, antivirals, antifungals
    "amoxicillin": "amoxicillin", "augmentin": "amoxicillin-clavulanate", "azithromycin": "azithromycin",
    "z-pack": "azithromycin", "zpack": "azithromycin", "z-pak": "azithromycin", "doxycycline": "doxycycline",
    "cephalexin": "cephalexin", "keflex": "cephalexin", "ciprofloxacin": "ciprofloxacin", "cipro": "ciprofloxacin",
    "nitrofurantoin": "nitrofurantoin", "macrobid": "nitrofurantoin", "penicillin": "penicillin",
    "clindamycin": "clindamycin", "metronidazole": "metronidazole", "flagyl": "metronidazole",
    "bactrim": "sulfamethoxazole-trimethoprim", "fluconazole": "fluconazole", "diflucan": "fluconazole",
    "valacyclovir": "valacyclovir", "valtrex": "valacyclovir", "acyclovir": "acyclovir",
    "tamiflu": "oseltamivir", "oseltamivir": "oseltamivir", "paxlovid": "nirmatrelvir-ritonavir",
    # mental health, sleep, ADHD
    "sertraline": "sertraline", "zoloft": "sertraline", "fluoxetine": "fluoxetine", "prozac": "fluoxetine",
    "escitalopram": "escitalopram", "lexapro": "escitalopram", "citalopram": "citalopram",
    "bupropion": "bupropion", "wellbutrin": "bupropion", "venlafaxine": "venlafaxine", "effexor": "venlafaxine",
    "duloxetine": "duloxetine", "cymbalta": "duloxetine", "trazodone": "trazodone", "mirtazapine": "mirtazapine",
    "buspirone": "buspirone", "alprazolam": "alprazolam", "xanax": "alprazolam", "lorazepam": "lorazepam",
    "ativan": "lorazepam", "clonazepam": "clonazepam", "klonopin": "clonazepam", "diazepam": "diazepam",
    "valium": "diazepam", "zolpidem": "zolpidem", "ambien": "zolpidem", "quetiapine": "quetiapine",
    "seroquel": "quetiapine", "aripiprazole": "aripiprazole", "abilify": "aripiprazole", "lithium": "lithium",
    "lamotrigine": "lamotrigine", "lamictal": "lamotrigine", "adderall": "amphetamine salts",
    "methylphenidate": "methylphenidate", "ritalin": "methylphenidate", "concerta": "methylphenidate",
    "vyvanse": "lisdexamfetamine",
    # pain, nerves, migraine, muscle
    "gabapentin": "gabapentin", "neurontin": "gabapentin", "pregabalin": "pregabalin", "lyrica": "pregabalin",
    "cyclobenzaprine": "cyclobenzaprine", "flexeril": "cyclobenzaprine", "tramadol": "tramadol",
    "oxycodone": "oxycodone", "hydrocodone": "hydrocodone", "codeine": "codeine", "morphine": "morphine",
    "sumatriptan": "sumatriptan", "imitrex": "sumatriptan", "rizatriptan": "rizatriptan", "topiramate": "topiramate",
    "naloxone": "naloxone", "narcan": "naloxone", "meloxicam": "meloxicam", "colchicine": "colchicine",
    "allopurinol": "allopurinol",
    # skin, hormones, other
    "tretinoin": "tretinoin", "retin-a": "tretinoin", "isotretinoin": "isotretinoin", "accutane": "isotretinoin",
    "benzoyl peroxide": "benzoyl peroxide", "hydrocortisone cream": "hydrocortisone",
    "birth control": "hormonal birth control", "estradiol": "estradiol", "progesterone": "progesterone",
    "finasteride": "finasteride", "minoxidil": "minoxidil", "sildenafil": "sildenafil", "viagra": "sildenafil",
    "tadalafil": "tadalafil", "cialis": "tadalafil", "methotrexate": "methotrexate",
    "hydroxychloroquine": "hydroxychloroquine", "plaquenil": "hydroxychloroquine", "humira": "adalimumab",
}

SUPPLEMENT_TERMS = {
    "melatonin": "melatonin", "vitamin d": "vitamin d", "vitamin b12": "vitamin b12", "b12": "vitamin b12",
    "vitamin c": "vitamin c", "iron supplement": "iron", "magnesium": "magnesium", "zinc": "zinc",
    "fish oil": "fish oil", "omega-3": "fish oil", "probiotic": "probiotic", "probiotics": "probiotic",
    "ashwagandha": "ashwagandha", "turmeric": "turmeric", "electrolytes": "electrolytes",
}

DRUG_CLASS_TERMS = {
    "antibiotic": "antibiotic (class)", "antibiotics": "antibiotic (class)",
    "antihistamine": "antihistamine (class)", "antihistamines": "antihistamine (class)",
    "nsaid": "nsaid (class)", "nsaids": "nsaid (class)", "antidepressant": "antidepressant (class)",
    "antidepressants": "antidepressant (class)", "antiviral": "antiviral (class)", "antivirals": "antiviral (class)",
    "steroid": "steroid (class)", "steroids": "steroid (class)", "decongestant": "decongestant (class)",
    "pain reliever": "pain reliever (class)", "pain relievers": "pain reliever (class)",
    "antacid": "antacid (class)", "antacids": "antacid (class)",
}

INCLUDE_SUPPLEMENTS = False
INCLUDE_DRUG_CLASSES = False

# Generic drug-name endings; a word must be at least MED_SUFFIX_MIN_LEN letters to count
MED_SUFFIXES = ["pril", "olol", "sartan", "statin", "azole", "prazole", "cillin", "mycin", "cycline",
                "floxacin", "tidine", "oxetine", "triptan", "zepam", "zolam", "gliptin", "gliflozin",
                "glutide", "dronate", "semide", "thiazide", "dipine", "afil", "tinib", "umab", "imab"]
MED_SUFFIX_MIN_LEN = 7
MED_SUFFIX_EXCLUDE = {"april"}

# Length split for the first participant message: "n_words" or "n_chars"
LENGTH_MEASURE = "n_words"

## 3. Matching functions

In [ ]:
APOSTROPHES = "'\u2019"
STOPWORDS = {"a", "an", "the", "this", "that", "these", "those", "my", "your", "his", "her", "their", "our",
             "its", "any", "some", "no", "of", "or", "and", "a", "it", "is", "be", "an", "for", "with", "from",
             "to", "in", "on", "what", "which", "other", "another", "same", "kind", "type", "sort", "serious",
             "bad", "possible", "chronic", "common", "rare", "having", "have", "has", "get", "got"}


def _strip_apostrophes(text):
    return re.sub(f"[{APOSTROPHES}]", "", text)


def _term_regex(term):
    """Whole-word pattern where spaces and hyphens are interchangeable."""
    parts = re.split(r"[\s\-]+", _strip_apostrophes(term).strip())
    return r"[\s\-]+".join(re.escape(p) for p in parts)


def build_term_pattern(terms, case_sensitive=False):
    """One pattern for a list of terms, longest first so 'type 2 diabetes' wins over 'diabetes'."""
    terms = sorted(set(terms), key=len, reverse=True)
    alt = "|".join(_term_regex(t) for t in terms)
    flags = 0 if case_sensitive else re.IGNORECASE
    return re.compile(rf"(?<!\w)({alt})(?!\w)", flags)


def _norm(s):
    return re.sub(r"[\s\-]+", " ", _strip_apostrophes(s).lower()).strip()


def find_diseases(text):
    """Return the sorted list of disease terms found in one piece of text."""
    if not isinstance(text, str) or not text.strip():
        return []
    t = _strip_apostrophes(text)          # so "crohn's" and "crohns" match the same way
    hits = {_norm(m.group(1)) for m in DISEASE_RE.finditer(t)}
    hits |= {m.group(1) for m in ACRONYM_RE.finditer(t)}
    for m in SUFFIX_RE.finditer(t):
        w = m.group(0).lower()
        if w not in SUFFIX_EXCLUDE:
            hits.add(w)
    for m in HEAD_RE.finditer(t):        # "sinus infection", "vitamin d deficiency"
        words = m.group("before").lower().split()
        while words and words[0] in STOPWORDS:
            words.pop(0)
        if words:
            hits.add(" ".join(words + [m.group("head").lower()]))
    return sorted(hits)


def find_meds(text):
    """Return the sorted list of standard medication names found in one piece of text."""
    if not isinstance(text, str) or not text.strip():
        return []
    t = _strip_apostrophes(text)
    found = {MED_LOOKUP[_norm(m.group(1))] for m in MED_RE.finditer(t)}
    for m in MED_SUFFIX_RE.finditer(t):
        w = m.group(0).lower()
        if len(w) >= MED_SUFFIX_MIN_LEN and w not in MED_SUFFIX_EXCLUDE and w not in MED_KNOWN:
            found.add(w)
    return sorted(found)


def build_patterns():
    """Compile every pattern from the Section 2 settings. Rerun after editing the lists."""
    global DISEASE_RE, ACRONYM_RE, SUFFIX_RE, HEAD_RE, MED_RE, MED_SUFFIX_RE, MED_LOOKUP, MED_KNOWN
    DISEASE_RE = build_term_pattern(DISEASE_TERMS)
    ACRONYM_RE = build_term_pattern(DISEASE_ACRONYMS, case_sensitive=True)
    SUFFIX_RE = re.compile(rf"\b[a-z]{{3,}}(?:{'|'.join(DISEASE_SUFFIXES)})\b", re.IGNORECASE)
    HEAD_RE = re.compile(rf"\b(?P<before>(?:[a-z0-9]+\s+){{1,2}})(?P<head>{'|'.join(DISEASE_HEAD_WORDS)})\b",
                         re.IGNORECASE)

    lookup = dict(MED_TERMS)
    if INCLUDE_SUPPLEMENTS:
        lookup.update(SUPPLEMENT_TERMS)
    if INCLUDE_DRUG_CLASSES:
        lookup.update(DRUG_CLASS_TERMS)
    MED_LOOKUP = {_norm(k): v for k, v in lookup.items()}
    MED_KNOWN = set(MED_LOOKUP) | set(lookup.values())
    MED_RE = build_term_pattern(list(lookup))
    MED_SUFFIX_RE = re.compile(rf"\b[a-z]+(?:{'|'.join(MED_SUFFIXES)})\b", re.IGNORECASE)


build_patterns()

### 3a. Check the matching on made-up text

Edit these sentences to see what the lists catch before running the real data.

In [ ]:
checks = [
    "do i have type 2 diabetes or is it just prediabetes",
    "my mom has Crohns and MS, could that be related",
    "i think its tendonitis from running",
    "I have a headache and a rash",                               # symptoms only -> nothing
    "is it ok to take Advil with my lisinopril",
    "Ibuprofen (Advil) or acetaminophen can help, or ask about a rosuvastatin alternative.",
    "a vitamin d deficiency can cause fatigue",
]
pd.DataFrame({"text": checks,
              "diseases": [find_diseases(t) for t in checks],
              "meds": [find_meds(t) for t in checks]})

## 4. Build the binary variables

In [ ]:
def has_text(x):
    return isinstance(x, str) and x.strip() != ""


out = pd.DataFrame({ID_COL: wide[ID_COL]})
audit = pd.DataFrame({ID_COL: wide[ID_COL]})

# ---- 1. Participant names a disease or condition (any participant turn) ----
human_any_text = wide[HUMAN_COLS].apply(lambda r: any(has_text(v) for v in r), axis=1)
audit["human_disease_terms"] = wide[HUMAN_COLS].apply(
    lambda r: "; ".join(sorted({d for v in r for d in find_diseases(v)})), axis=1)
out["human_mentions_disease"] = np.where(
    ~human_any_text, np.nan, (audit["human_disease_terms"] != "").astype(int))

# ---- 2. More than one participant turn (counted from the text columns) ----
n_human = wide[HUMAN_COLS].apply(lambda r: sum(has_text(v) for v in r), axis=1).replace(0, np.nan)
audit["n_human_turns"] = n_human
out["human_multi_turn"] = np.where(n_human.isna(), np.nan, (n_human > 1).astype(int))

# Cross-check against the count made in the earlier scripts
diff = (n_human.fillna(0) != wide["n_human_turns"].fillna(0)).sum()
print(f"Participants where the text count differs from n_human_turns: {diff}")

# ---- 3. AI names more than one distinct medication (all AI turns, including ai_0) ----
ai_meds = wide[AI_COLS].apply(lambda r: sorted({m for v in r for m in find_meds(v)}), axis=1)
audit["ai_med_names"] = ai_meds.map("; ".join)
audit["ai_n_meds"] = ai_meds.map(len)
out["ai_multi_med"] = np.where(audit["ai_n_meds"] == 0, np.nan, (audit["ai_n_meds"] > 1).astype(int))
audit["ai_any_med"] = (audit["ai_n_meds"] > 0).astype(int)

# ---- 4. Long vs short first participant message (median split, measured from the text) ----
def text_length(t):
    if not has_text(t):
        return np.nan
    return len(t.split()) if LENGTH_MEASURE == "n_words" else len(t.strip())

length = wide["human_1"].map(text_length)
median_len = length.median()
audit[f"human_1_{LENGTH_MEASURE}"] = length
out["human_1_long"] = np.where(length.isna(), np.nan, (length > median_len).astype(int))

out = out.astype({c: "Int64" for c in out.columns if c != ID_COL})
print(f"Median first-message length: {median_len} ({LENGTH_MEASURE.replace('n_', '')})")
out.describe(include="all").loc[["count", "mean"]]

## 5. Check the results

In [ ]:
summary = pd.DataFrame({
    v: out[v].value_counts(dropna=False).reindex([1, 0, pd.NA], fill_value=0).values
    for v in out.columns if v != ID_COL
}, index=["1", "0", "missing"]).T
summary

In [ ]:
# Which terms drove the disease and medication codes? Look for false matches here.
terms = audit["human_disease_terms"].str.split("; ").explode().replace("", np.nan).dropna()
meds = audit["ai_med_names"].str.split("; ").explode().replace("", np.nan).dropna()
display(terms.value_counts().to_frame("n_participants_disease_term"))
display(meds.value_counts().to_frame("n_participants_med_name"))

## 6. Save

In [ ]:
codebook = pd.DataFrame([
    ("pid", "Participant ID (de-identified)", "", ""),
    ("human_mentions_disease", "Participant names a disease or condition in any of their turns",
     "1 = yes, 0 = no", "Missing if no participant text. Word lists in Section 2; symptoms alone do not count."),
    ("human_multi_turn", "Participant wrote more than one turn",
     "1 = more than one turn, 0 = one turn", "Counted from the filled human_ text columns. Missing if no participant turns."),
    ("ai_multi_med", "AI names more than one distinct medication across its turns",
     "1 = two or more, 0 = exactly one", "Missing when the AI names no medication. Brand and generic count once. "
     f"Supplements included: {INCLUDE_SUPPLEMENTS}. Drug classes included: {INCLUDE_DRUG_CLASSES}."),
    ("human_1_long", "First participant message is longer than the median",
     "1 = above median, 0 = at or below median",
     f"Measured from the human_1 text. Measure: {LENGTH_MEASURE}. Median = {median_len}. Ties at the median are coded 0."),
], columns=["variable", "definition", "coding", "notes"])

with pd.ExcelWriter(OUT_FILE) as xw:
    out.to_excel(xw, sheet_name="binary_vars", index=False)
    audit.to_excel(xw, sheet_name="audit", index=False)
    codebook.to_excel(xw, sheet_name="codebook", index=False)

wb = load_workbook(OUT_FILE)
for ws in wb.worksheets:
    ws.freeze_panes = "B2"
    for cell in ws[1]:
        cell.font = Font(bold=True)
    for col in ws.columns:
        width = min(60, max(10, *(len(str(c.value)) for c in col if c.value is not None)))
        ws.column_dimensions[col[0].column_letter].width = width + 2
for row in wb["codebook"].iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical="top")
wb.save(OUT_FILE)
print(f"Saved {OUT_FILE}")

## 7. Counts and pie charts

One pie per variable, showing how many participants fall in each group. Participants coded missing are left out of the pie and listed under it. Following Healogix chart standards, the slices use two blues: dark blue for the group coded 1 and light blue for the group coded 0. Every slice is also labeled with its group name, count and percentage, so the chart doesn't rely on color alone.

The chart is also saved as a PNG in `output/06_Q4_3_binary-categories/`.

In [ ]:
# Counts table
counts = pd.DataFrame({
    "variable": [v for v in out.columns if v != ID_COL],
    "n_1": [int((out[v] == 1).sum()) for v in out.columns if v != ID_COL],
    "n_0": [int((out[v] == 0).sum()) for v in out.columns if v != ID_COL],
    "n_missing": [int(out[v].isna().sum()) for v in out.columns if v != ID_COL],
})
counts["n_coded"] = counts["n_1"] + counts["n_0"]
counts["pct_1"] = (100 * counts["n_1"] / counts["n_coded"].replace(0, np.nan)).round(1)
counts

In [ ]:
import matplotlib.pyplot as plt

# Plain-language titles and group names for each variable: (title, label for 1, label for 0)
PIE_LABELS = {
    "human_mentions_disease": ("Participant named a disease", "Yes", "No"),
    "human_multi_turn":       ("Number of participant turns", "More than one", "One"),
    "ai_multi_med":           ("Medications the AI named\n(among conversations naming any)", "Two or more", "One"),
    "human_1_long":           (f"Length of first message\n(median = {median_len:g} {LENGTH_MEASURE.replace('n_', '')})",
                               "Above median", "At or below\nmedian"),
}
COLOR_1 = "#1F4E79"   # dark blue
COLOR_0 = "#9DC3E6"   # light blue
INK = "#1A1A1A"
MUTED = "#5F6B7A"

FIG_DIR = PROJECT_ROOT / "output" / "06_Q4_3_binary-categories"
FIG_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(10, 9.5), facecolor="white")
for ax, (var, (title, lab1, lab0)) in zip(axes.flat, PIE_LABELS.items()):
    row = counts.set_index("variable").loc[var]
    n1, n0, n_miss = int(row["n_1"]), int(row["n_0"]), int(row["n_missing"])
    ax.set_title(title, fontsize=12, color=INK, pad=10)
    if n1 + n0 == 0:
        ax.text(0.5, 0.5, "No coded participants", ha="center", va="center", color=MUTED, transform=ax.transAxes)
        ax.axis("off")
        continue
    total = n1 + n0
    sizes, labels, colors, text_colors = [], [], [], []
    for n, lab, col, tc in [(n1, lab1, COLOR_1, "white"), (n0, lab0, COLOR_0, INK)]:
        if n > 0:
            sizes.append(n)
            labels.append(f"{lab}\nn = {n} ({100 * n / total:.0f}%)")
            colors.append(col)
            text_colors.append(tc)
    wedges, texts = ax.pie(sizes, labels=labels, colors=colors, startangle=90, counterclock=False,
                           labeldistance=0.55, wedgeprops={"edgecolor": "white", "linewidth": 2},
                           textprops={"fontsize": 10, "ha": "center", "va": "center"})
    for t, tc in zip(texts, text_colors):
        t.set_color(tc)
    ax.set_aspect("equal")
    note = f"N = {total}" + (f"  |  {n_miss} missing, not shown" if n_miss else "")
    ax.text(0.5, -0.04, note, ha="center", va="top", fontsize=9, color=MUTED, transform=ax.transAxes)

fig.suptitle("Q4 Pulse 3: conversation groups", fontsize=14, color=INK, y=0.995)
fig.tight_layout()
fig.savefig(FIG_DIR / "binary_pies.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()